In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

# =========================================================
# 1. 파일 경로
# =========================================================
path_2017 = r"D:\dataset\2017.csv"
path_2018 = r"D:\dataset\2018.csv"

In [2]:
# =========================================================
# 2. 2018 -> 2017 컬럼명 매핑
# =========================================================
rename_2018_to_2017 = {
    "Dst Port": "Destination Port",
    "Tot Fwd Pkts": "Total Fwd Packets",
    "Tot Bwd Pkts": "Total Backward Packets",
    "TotLen Fwd Pkts": "Total Length of Fwd Packets",
    "TotLen Bwd Pkts": "Total Length of Bwd Packets",
    "Fwd Pkt Len Max": "Fwd Packet Length Max",
    "Fwd Pkt Len Min": "Fwd Packet Length Min",
    "Fwd Pkt Len Mean": "Fwd Packet Length Mean",
    "Fwd Pkt Len Std": "Fwd Packet Length Std",
    "Bwd Pkt Len Max": "Bwd Packet Length Max",
    "Bwd Pkt Len Min": "Bwd Packet Length Min",
    "Bwd Pkt Len Mean": "Bwd Packet Length Mean",
    "Bwd Pkt Len Std": "Bwd Packet Length Std",
    "Flow Byts/s": "Flow Bytes/s",
    "Flow Pkts/s": "Flow Packets/s",
    "Fwd IAT Tot": "Fwd IAT Total",
    "Bwd IAT Tot": "Bwd IAT Total",
    "Fwd Header Len": "Fwd Header Length",
    "Bwd Header Len": "Bwd Header Length",
    "Fwd Pkts/s": "Fwd Packets/s",
    "Bwd Pkts/s": "Bwd Packets/s",
    "Pkt Len Min": "Min Packet Length",
    "Pkt Len Max": "Max Packet Length",
    "Pkt Len Mean": "Packet Length Mean",
    "Pkt Len Std": "Packet Length Std",
    "Pkt Len Var": "Packet Length Variance",
    "FIN Flag Cnt": "FIN Flag Count",
    "SYN Flag Cnt": "SYN Flag Count",
    "RST Flag Cnt": "RST Flag Count",
    "PSH Flag Cnt": "PSH Flag Count",
    "ACK Flag Cnt": "ACK Flag Count",
    "URG Flag Cnt": "URG Flag Count",
    "ECE Flag Cnt": "ECE Flag Count",
    "Pkt Size Avg": "Average Packet Size",
    "Fwd Seg Size Avg": "Avg Fwd Segment Size",
    "Bwd Seg Size Avg": "Avg Bwd Segment Size",
    "Subflow Fwd Pkts": "Subflow Fwd Packets",
    "Subflow Fwd Byts": "Subflow Fwd Bytes",
    "Subflow Bwd Pkts": "Subflow Bwd Packets",
    "Subflow Bwd Byts": "Subflow Bwd Bytes",
    "Init Fwd Win Byts": "Init_Win_bytes_forward",
    "Init Bwd Win Byts": "Init_Win_bytes_backward",
}

In [3]:
# =========================================================
# 3. 라벨 정리
# =========================================================
def clean_label(x):
    x = str(x).strip()
    x = x.replace("�", "-")
    x = x.replace("–", "-")

    x = x.replace("Web Attack � Brute Force", "Web Attack - Brute Force")
    x = x.replace("Web Attack � XSS", "Web Attack - XSS")
    x = x.replace("Web Attack � Sql Injection", "Web Attack - Sql Injection")

    if x.lower() == "benign":
        return "BENIGN"

    return x


In [4]:
# =========================================================
# 4. 최종 라벨 통합 기준
# =========================================================
def unify_label(label):
    label = str(label).strip()
    label = label.replace("�", "-")
    label = label.replace("–", "-")

    if label.lower() == "benign":
        return "BENIGN"

    elif label in ["FTP-Patator", "FTP-BruteForce"]:
        return "FTP-BruteForce"

    elif label in ["SSH-Patator", "SSH-Bruteforce"]:
        return "SSH-Bruteforce"

    elif label in ["DoS slowloris", "DoS attacks-Slowloris"]:
        return "DoS Slowloris"

    elif label in ["DoS Slowhttptest", "DoS attacks-SlowHTTPTest"]:
        return "DoS SlowHTTPTest"

    elif label in ["DoS Hulk", "DoS attacks-Hulk"]:
        return "DoS Hulk"

    elif label in ["DoS GoldenEye", "DoS attacks-GoldenEye"]:
        return "DoS GoldenEye"

    elif label == "DDoS":
        return "DDoS"

    elif label == "DDOS attack-HOIC":
        return "DDOS attack-HOIC"

    elif label in [
        "Web Attack - Brute Force",
        "Web Attack - XSS",
        "Web Attack - Sql Injection",
        "Web Attack - SQL Injection",
        "Web Attack- Brute Force",
        "Web Attack- XSS",
        "Web Attack- Sql Injection"
    ]:
        return "Web Attack"

    else:
        return label

In [5]:
# =========================================================
# 5. 데이터 로드
# =========================================================
df_2017 = pd.read_csv(path_2017, low_memory=False)
df_2018 = pd.read_csv(path_2018, low_memory=False)

# 컬럼 공백 제거
df_2017.columns = df_2017.columns.str.strip()
df_2018.columns = df_2018.columns.str.strip()

# 2018 컬럼명을 2017 스타일로 맞춤
df_2018.rename(columns=rename_2018_to_2017, inplace=True)

# 불필요 컬럼 제거
drop_cols = ["Protocol", "Timestamp", "Fwd Header Length.1"]
df_2017.drop(columns=[c for c in drop_cols if c in df_2017.columns], inplace=True)
df_2018.drop(columns=[c for c in drop_cols if c in df_2018.columns], inplace=True)

# 라벨 정리
df_2017["Label"] = df_2017["Label"].apply(clean_label).apply(unify_label)
df_2018["Label"] = df_2018["Label"].apply(clean_label).apply(unify_label)


In [14]:
print("2017 라벨 분포:")
print(df_2017["Label"].value_counts(), "\n")

print("2018 라벨 분포:")
print(df_2018["Label"].value_counts(), "\n")

2017 라벨 분포:
Label
BENIGN              201666
PortScan            100833
DoS Hulk            100833
DDoS                100833
DoS GoldenEye        10293
FTP-BruteForce        7938
SSH-Bruteforce        5897
DoS Slowloris         5796
DoS SlowHTTPTest      5499
Web Attack            2180
Bot                   1966
Name: count, dtype: int64 

2018 라벨 분포:
Label
FTP-BruteForce      193360
SSH-Bruteforce      187589
BENIGN              160253
Bot                 153040
Infilteration        93063
DoS SlowHTTPTest     50432
DDOS attack-HOIC     44334
DoS GoldenEye        38532
DoS Hulk             36145
DoS Slowloris        10213
Name: count, dtype: int64 



In [16]:
# =========================================================
# 6. 공통 컬럼만 사용
# =========================================================
common_cols = list(set(df_2017.columns).intersection(set(df_2018.columns)))
common_cols = [c for c in common_cols if c != "Label"] + ["Label"]

df_2017 = df_2017[common_cols].copy()
df_2018 = df_2018[common_cols].copy()

print("공통 컬럼 수:", len(common_cols))
print("예시 공통 컬럼 일부:", common_cols[:10])

공통 컬럼 수: 70
예시 공통 컬럼 일부: ['Fwd IAT Min', 'ACK Flag Count', 'Fwd Packet Length Min', 'FIN Flag Count', 'Flow Duration', 'Max Packet Length', 'Subflow Fwd Bytes', 'Active Mean', 'Bwd Header Length', 'Bwd IAT Max']


In [20]:
# =========================================================
# 7. 기본 전처리
# =========================================================
def preprocess(df):
    df = df.copy()

    df.replace([np.inf, -np.inf], np.nan, inplace=True)

    if "Flow Duration" in df.columns:
        df = df[df["Flow Duration"] >= 0]

    df.dropna(subset=["Label"], inplace=True)

    # 비수치형 컬럼 제거
    non_numeric_cols = [c for c in df.columns if c != "Label" and not pd.api.types.is_numeric_dtype(df[c])]
    if non_numeric_cols:
        print("제거된 비수치형 컬럼:", non_numeric_cols)
        df.drop(columns=non_numeric_cols, inplace=True)

    # 숫자형 결측치 중앙값 대체
    feature_cols = [c for c in df.columns if c != "Label"]
    for col in feature_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        df[col] = df[col].fillna(df[col].median())

    return df

df_2017 = preprocess(df_2017)
df_2018 = preprocess(df_2018)
# 공통 특성 다시 한번 정렬
final_feature_cols = list(set(df_2017.columns).intersection(set(df_2018.columns)))
final_feature_cols = [c for c in final_feature_cols if c != "Label"]

df_2017 = df_2017[final_feature_cols + ["Label"]].copy()
df_2018 = df_2018[final_feature_cols + ["Label"]].copy()

print("\n최종 2017 shape:", df_2017.shape)
print("최종 2018 shape:", df_2018.shape)
print("최종 feature 개수:", len(final_feature_cols))



최종 2017 shape: (543725, 70)
최종 2018 shape: (966961, 70)
최종 feature 개수: 69


In [22]:
# =========================================================
# 8. 2017 train / 2018 test
# =========================================================
X_train = df_2017.drop(columns=["Label"])
y_train = df_2017["Label"]

X_test = df_2018.drop(columns=["Label"])
y_test = df_2018["Label"]

# 2017에 없는 클래스는 2018 테스트에서 제외
known_classes = sorted(y_train.unique())
mask = y_test.isin(known_classes)

X_test = X_test[mask].copy()
y_test = y_test[mask].copy()

print("\n2017 학습 클래스:")
print(sorted(y_train.unique()))

print("\n2018 테스트 클래스(평가 대상):")
print(sorted(y_test.unique()))


2017 학습 클래스:
['BENIGN', 'Bot', 'DDoS', 'DoS GoldenEye', 'DoS Hulk', 'DoS SlowHTTPTest', 'DoS Slowloris', 'FTP-BruteForce', 'PortScan', 'SSH-Bruteforce', 'Web Attack']

2018 테스트 클래스(평가 대상):
['BENIGN', 'Bot', 'DoS GoldenEye', 'DoS Hulk', 'DoS SlowHTTPTest', 'DoS Slowloris', 'FTP-BruteForce', 'SSH-Bruteforce']


In [24]:
# =========================================================
# 9. 라벨 인코딩
# =========================================================
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

print("\nLabelEncoder 클래스 순서:")
print(list(le.classes_))

joblib.dump(le, "label_encoder_2017_train_2018_test.pkl")


LabelEncoder 클래스 순서:
['BENIGN', 'Bot', 'DDoS', 'DoS GoldenEye', 'DoS Hulk', 'DoS SlowHTTPTest', 'DoS Slowloris', 'FTP-BruteForce', 'PortScan', 'SSH-Bruteforce', 'Web Attack']


['label_encoder_2017_train_2018_test.pkl']

In [26]:
# =========================================================
# 10. 스케일링
# =========================================================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

joblib.dump(scaler, "scaler_2017_train_2018_test.pkl")

['scaler_2017_train_2018_test.pkl']

In [28]:
# =========================================================
# 11. RandomForest
# =========================================================
print("\n================ RandomForest (2017 train -> 2018 test) ================")

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced_subsample"
)

rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("RF Accuracy:", accuracy_score(y_test, y_pred_rf))
print("\nRF Classification Report:")
print(classification_report(y_test, y_pred_rf, zero_division=0))

joblib.dump(rf, "rf_2017_train_2018_test.pkl")


================ RandomForest (2017 train -> 2018 test) ================
RF Accuracy: 0.21729125179009698

RF Classification Report:
                  precision    recall  f1-score   support

          BENIGN       0.20      0.99      0.33    160253
             Bot       0.00      0.00      0.00    153040
   DoS GoldenEye       0.99      0.33      0.49     38532
        DoS Hulk       0.32      0.02      0.03     36145
DoS SlowHTTPTest       0.00      0.00      0.00     50432
   DoS Slowloris       1.00      0.79      0.88     10213
  FTP-BruteForce       0.00      0.00      0.00    193360
        PortScan       0.00      0.00      0.00         0
  SSH-Bruteforce       1.00      0.00      0.00    187589
      Web Attack       0.00      0.00      0.00         0

        accuracy                           0.22    829564
       macro avg       0.35      0.21      0.17    829564
    weighted avg       0.34      0.22      0.10    829564



['rf_2017_train_2018_test.pkl']

In [29]:
# =========================================================
# 12. XGBoost
# =========================================================
print("\n================ XGBoost (2017 train -> 2018 test) ================")

xgb = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softmax",
    num_class=len(le.classes_),
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)

xgb.fit(X_train, y_train_enc)
y_pred_xgb_enc = xgb.predict(X_test)
y_pred_xgb = le.inverse_transform(y_pred_xgb_enc)

print("XGB Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("\nXGB Classification Report:")
print(classification_report(y_test, y_pred_xgb, zero_division=0))

joblib.dump(xgb, "xgb_2017_train_2018_test.pkl")



================ XGBoost (2017 train -> 2018 test) ================
XGB Accuracy: 0.22711207333008665

XGB Classification Report:
                  precision    recall  f1-score   support

          BENIGN       0.28      0.98      0.44    160253
             Bot       1.00      0.00      0.01    153040
   DoS GoldenEye       1.00      0.18      0.31     38532
        DoS Hulk       0.97      0.40      0.57     36145
DoS SlowHTTPTest       0.00      0.00      0.00     50432
   DoS Slowloris       1.00      0.97      0.99     10213
  FTP-BruteForce       0.00      0.00      0.00    193360
        PortScan       0.00      0.00      0.00         0
  SSH-Bruteforce       0.10      0.00      0.00    187589
      Web Attack       0.00      0.00      0.00         0

        accuracy                           0.23    829564
       macro avg       0.43      0.25      0.23    829564
    weighted avg       0.36      0.23      0.14    829564



['xgb_2017_train_2018_test.pkl']

In [30]:
# =========================================================
# 13. DNN
# =========================================================
print("\n================ DNN (2017 train -> 2018 test) ================")

num_classes = len(le.classes_)
y_train_cat = to_categorical(y_train_enc, num_classes=num_classes)
y_test_cat = to_categorical(y_test_enc, num_classes=num_classes)

dnn = Sequential([
    Input(shape=(X_train_scaled.shape[1],)),
    Dense(256, activation="relu"),
    Dropout(0.3),
    Dense(128, activation="relu"),
    Dropout(0.3),
    Dense(64, activation="relu"),
    Dense(num_classes, activation="softmax")
])

dnn.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

dnn.fit(
    X_train_scaled,
    y_train_cat,
    validation_split=0.2,
    epochs=30,
    batch_size=512,
    callbacks=[early_stop],
    verbose=1
)

y_pred_dnn_prob = dnn.predict(X_test_scaled)
y_pred_dnn_enc = np.argmax(y_pred_dnn_prob, axis=1)
y_pred_dnn = le.inverse_transform(y_pred_dnn_enc)

print("DNN Accuracy:", accuracy_score(y_test, y_pred_dnn))
print("\nDNN Classification Report:")
print(classification_report(y_test, y_pred_dnn, zero_division=0))

dnn.save("dnn_2017_train_2018_test.keras")


================ DNN (2017 train -> 2018 test) ================
Epoch 1/30
850/850 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.8885 - loss: 0.3941 - val_accuracy: 0.9816 - val_loss: 0.0558
Epoch 2/30
850/850 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9778 - loss: 0.0702 - val_accuracy: 0.9852 - val_loss: 0.0445
Epoch 3/30
850/850 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9826 - loss: 0.0543 - val_accuracy: 0.9866 - val_loss: 0.0405
Epoch 4/30
850/850 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9849 - loss: 0.0471 - val_accuracy: 0.9870 - val_loss: 0.0385
Epoch 5/30
850/850 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9855 - loss: 0.0445 - val_accuracy: 0.9849 - val_loss: 0.0442
Epoch 6/30
850/850 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9860 - loss: 0.0428 - val_accuracy: 0.9901 - val_loss: 0.0326
Epoch 7/30
850/850 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9871 - loss: 0.0398 - val_accuracy: 0.9904 - val_loss: 0.0307
Epoch 8/30
850/850 ━━━━━━━━━━━━━━━━━━━━

In [31]:
# =========================================================
# 14. 결과 요약
# =========================================================
results = pd.DataFrame({
    "Model": ["RandomForest", "XGBoost", "DNN"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_xgb),
        accuracy_score(y_test, y_pred_dnn)
    ]
})

print("\n================ 결과 요약 ================")
print(results.sort_values(by="Accuracy", ascending=False))


================ 결과 요약 ================
          Model  Accuracy
2           DNN  0.394354
1       XGBoost  0.227112
0  RandomForest  0.217291
